# 36 - QUAL-002 / QUAL-004: Held-out con Ground Truth y evaluación Dice/IoU — v3 self-contained

Este notebook corrige el bloqueo de la versión anterior:

- La copia del held-out ya funcionaba usando `/content/pfi_qual_work`.
- El nuevo problema era que en Colab no aparecía `scripts/evaluate_model.py`.
- Esta versión **no depende de `scripts/evaluate_model.py`**: si no lo encuentra, ejecuta un evaluador interno.
- Si encuentra `pfi_ai_service/model_architectures.py`, usa `build_checkpoint_model()` del repo.
- Si no lo encuentra, te avisa con un error claro para montar/clonar el AI Module.

Advertencia metodológica: si `selection_mode != clean_test_from_split`, reportar como **evaluación preliminar con ground truth**, no como test limpio final por paciente.


In [1]:
# ============================================
# 0) Montar Google Drive e instalar dependencias
# ============================================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install SimpleITK pydicom nibabel pandas pillow


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.4 MB/s eta 0:00:00


In [2]:
# ============================================
# 1) Configuración principal
# ============================================
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, csv, re, time, textwrap, zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive")
PFI_ROOT = DRIVE_ROOT / "PFI_MVP"

SAGITTAL_CKPT = PFI_ROOT / "models/final/sagittal_spider_multiclass_final_best.pt"
AXIAL_CKPT    = PFI_ROOT / "models/final/axial_t2_alkafri_final_best.pt"

# Salidas finales en Drive, solo evidencia chica
DRIVE_QUAL_ROOT = PFI_ROOT / "qual"
DRIVE_REPORTS_ROOT = DRIVE_QUAL_ROOT / "reports"
DRIVE_DOCS_ROOT = PFI_ROOT / "docs"

# Workdir local para evitar errores de Drive al copiar datasets
LOCAL_WORK_ROOT = Path("/content/pfi_qual_work")
LOCAL_HELDOUT_ROOT = LOCAL_WORK_ROOT / "heldout"
LOCAL_REPORTS_ROOT = LOCAL_WORK_ROOT / "reports"
LOCAL_DOCS_ROOT = LOCAL_WORK_ROOT / "docs"

SAG_TEST_DIR = LOCAL_HELDOUT_ROOT / "sagittal"
AX_TEST_DIR  = LOCAL_HELDOUT_ROOT / "axial"

SAG_IMAGES_OUT = SAG_TEST_DIR / "images"
SAG_MASKS_OUT  = SAG_TEST_DIR / "masks"
AX_IMAGES_OUT  = AX_TEST_DIR / "images"
AX_MASKS_OUT   = AX_TEST_DIR / "masks"

for d in [
    LOCAL_WORK_ROOT, LOCAL_REPORTS_ROOT, LOCAL_DOCS_ROOT,
    SAG_IMAGES_OUT, SAG_MASKS_OUT, AX_IMAGES_OUT, AX_MASKS_OUT,
    DRIVE_REPORTS_ROOT, DRIVE_DOCS_ROOT
]:
    d.mkdir(parents=True, exist_ok=True)

CLEAR_EXISTING_LOCAL_HELDOUT = True
TARGET_SIZE = 256

print("PFI_ROOT:", PFI_ROOT, "| exists:", PFI_ROOT.exists())
print("SAGITTAL_CKPT:", SAGITTAL_CKPT, "| exists:", SAGITTAL_CKPT.exists())
print("AXIAL_CKPT:", AXIAL_CKPT, "| exists:", AXIAL_CKPT.exists())
print("LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)
print("SAG_TEST_DIR:", SAG_TEST_DIR)
print("AX_TEST_DIR:", AX_TEST_DIR)


PFI_ROOT: /content/drive/MyDrive/PFI_MVP | exists: True
SAGITTAL_CKPT: /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt | exists: True
AXIAL_CKPT: /content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt | exists: True
LOCAL_WORK_ROOT: /content/pfi_qual_work
SAG_TEST_DIR: /content/pfi_qual_work/heldout/sagittal
AX_TEST_DIR: /content/pfi_qual_work/heldout/axial


In [3]:
# ============================================
# 2) Utilidades comunes + copia robusta
# ============================================
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def reset_dir(path: Path):
    if path.exists() and CLEAR_EXISTING_LOCAL_HELDOUT:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def remount_drive_if_needed():
    try:
        from google.colab import drive as colab_drive
        print("Intentando force_remount de Google Drive...")
        colab_drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print("No pude remontar automáticamente:", repr(e))


def copy_file_robust(src: Path, dst: Path, retries: int = 3, sleep_s: float = 2.0):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            shutil.copyfile(src, dst)
            return dst
        except OSError as e:
            last_error = e
            print(f"Copy failed attempt {attempt}/{retries}: {src} -> {dst} | {repr(e)}")
            if getattr(e, "errno", None) == 107 or "Transport endpoint" in str(e):
                remount_drive_if_needed()
            time.sleep(sleep_s)
        except Exception as e:
            last_error = e
            print(f"Copy failed attempt {attempt}/{retries}: {src} -> {dst} | {repr(e)}")
            time.sleep(sleep_s)

    raise RuntimeError(f"No se pudo copiar {src} -> {dst}. Último error: {repr(last_error)}")


def copy_pair(image_src: Path, mask_src: Path, image_dst: Path, mask_dst: Path):
    copy_file_robust(image_src, image_dst)
    copy_file_robust(mask_src, mask_dst)


def print_tree_counts():
    print("Sagital images:", len(list(SAG_IMAGES_OUT.glob("*"))))
    print("Sagital masks:", len(list(SAG_MASKS_OUT.glob("*"))))
    print("Axial images:", len(list(AX_IMAGES_OUT.glob("*"))))
    print("Axial masks:", len(list(AX_MASKS_OUT.glob("*"))))


## 3) Buscar splits/inventarios existentes

Esta celda intenta encontrar archivos de split, inventario, overview, train/validation/test, etc. Sirve para documentar si el held-out es limpio o preliminar.


In [4]:
# ============================================
# 3) Buscar archivos candidatos de split / inventario
# ============================================
patterns = [
    "*split*", "*overview*", "*holdout*", "*test*", "*train*", "*validation*", "*val*",
    "*pairing*", "*inventory*", "*manifest*"
]
valid_suffixes = {".csv", ".json", ".txt", ".parquet", ".tsv"}

hits = []
for pat in patterns:
    for p in PFI_ROOT.rglob(pat):
        if p.is_file() and p.suffix.lower() in valid_suffixes:
            normalized = str(p).replace("\\", "/").lower()
            if "/qual/heldout/" in normalized:
                continue
            hits.append(p)

hits = sorted(set(hits), key=lambda p: str(p).lower())
print("Archivos candidatos de split/inventario:", len(hits))
for i, p in enumerate(hits[:250]):
    print(f"{i:03d} | {p}")

split_scan_path = LOCAL_REPORTS_ROOT / "qual-002-split-candidates.txt"
split_scan_path.write_text("\n".join(str(p) for p in hits), encoding="utf-8")
print("\nGuardado local:", split_scan_path)


Archivos candidatos de split/inventario: 84
000 | /content/drive/MyDrive/PFI_MVP/data/SPIDER/overview.csv
001 | /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-split-candidates.txt
002 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_axial_t2_final_training_clean_report.json
003 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_test_metrics.json
004 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_training_history.csv
005 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_val_metrics.json
006 | /content/drive/MyDrive/PFI_MVP/results/E11_axial_class_mapping_final_report/E11_axial_class_distribution_by_split.csv
007 | /content/drive/MyDrive/PFI_MVP/results/E12_sagittal_final_training_clean/E12_json_metrics_inventory.csv
008 | /content/drive/MyDrive/PFI_MVP/results/E15_backend_mvp_translation/E15_service_file_inventory.csv
009 | /content/drive/MyDrive/PFI_MVP/results/E

In [5]:
# ============================================
# 3b) Preview de los primeros CSV/JSON relevantes
# ============================================
def preview_table_file(path: Path, max_rows=5):
    print("\n" + "=" * 100)
    print(path)
    try:
        if path.suffix.lower() in [".csv", ".tsv"]:
            sep = "\t" if path.suffix.lower() == ".tsv" else ","
            df = pd.read_csv(path, sep=sep)
            print("shape:", df.shape)
            print("columns:", list(df.columns)[:30])
            for col in df.columns:
                lc = col.lower()
                if lc in ["split", "subset", "fold", "set", "phase", "partition"]:
                    print(f"value_counts[{col}]:")
                    print(df[col].value_counts(dropna=False).head(20))
            print(df.head(max_rows).to_string(index=False))
        elif path.suffix.lower() == ".json":
            text = path.read_text(encoding="utf-8", errors="ignore")
            obj = json.loads(text)
            if isinstance(obj, dict):
                print("json keys:", list(obj.keys())[:30])
            elif isinstance(obj, list):
                print("json list len:", len(obj))
                print("first item:", obj[0] if obj else None)
    except Exception as e:
        print("No pude previsualizar:", repr(e))

for p in hits[:25]:
    preview_table_file(p)



/content/drive/MyDrive/PFI_MVP/data/SPIDER/overview.csv
shape: (447, 39)
columns: ['new_file_name', 'num_vertebrae', 'num_discs', 'sex', 'birth_date', 'subset', 'AngioFlag', 'BodyPartExamined', 'DeviceSerialNumber', 'EchoNumbers', 'EchoTime', 'EchoTrainLength', 'FlipAngle', 'ImagedNucleus', 'ImagingFrequency', 'InPlanePhaseEncodingDirection', 'MRAcquisitionType', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturerModelName', 'NumberOfPhaseEncodingSteps', 'PercentPhaseFieldOfView', 'PercentSampling', 'PhotometricInterpretation', 'PixelBandwidth', 'PixelSpacing', 'RepetitionTime', 'SAR', 'SamplesPerPixel', 'ScanningSequence']
value_counts[subset]:
subset
training      360
validation     87
Name: count, dtype: int64
new_file_name  num_vertebrae  num_discs sex  birth_date   subset AngioFlag BodyPartExamined  DeviceSerialNumber  EchoNumbers  EchoTime  EchoTrainLength  FlipAngle ImagedNucleus  ImagingFrequency InPlanePhaseEncodingDirection MRAcquisitionType  MagneticFieldStrength        

## 4) Verificar pares sagitales SPIDER

In [6]:
# ============================================
# 4) Pares sagitales SPIDER
# ============================================
SAG_IMAGES_SRC = PFI_ROOT / "data/SPIDER/images/images"
SAG_MASKS_SRC = PFI_ROOT / "data/SPIDER/masks/masks"

print("SAG_IMAGES_SRC:", SAG_IMAGES_SRC, "| exists:", SAG_IMAGES_SRC.exists())
print("SAG_MASKS_SRC:", SAG_MASKS_SRC, "| exists:", SAG_MASKS_SRC.exists())

def find_sag_mask_for_image(img: Path):
    exact = SAG_MASKS_SRC / img.name
    if exact.exists():
        return exact

    candidates = [p for p in SAG_MASKS_SRC.glob(img.stem + "*") if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]

    case_id = img.name.split("_")[0]
    candidates = [p for p in SAG_MASKS_SRC.glob(case_id + "*") if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]

    return None

sag_pairs = []
missing_sag_masks = []
for img in sorted(SAG_IMAGES_SRC.glob("*.mha")):
    if "_SPACE" in img.name:
        continue
    mask = find_sag_mask_for_image(img)
    if mask is not None and mask.exists():
        sag_pairs.append((img, mask))
    else:
        missing_sag_masks.append(img)

print("Pares sagitales encontrados:", len(sag_pairs))
print("Imágenes sagitales sin máscara detectada:", len(missing_sag_masks))
for img, mask in sag_pairs[:20]:
    print(img.name, "|", mask.name)


SAG_IMAGES_SRC: /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images | exists: True
SAG_MASKS_SRC: /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks | exists: True
Pares sagitales encontrados: 406
Imágenes sagitales sin máscara detectada: 0
100_t1.mha | 100_t1.mha
100_t2.mha | 100_t2.mha
101_t1.mha | 101_t1.mha
101_t2.mha | 101_t2.mha
104_t1.mha | 104_t1.mha
104_t2.mha | 104_t2.mha
105_t1.mha | 105_t1.mha
105_t2.mha | 105_t2.mha
106_t1.mha | 106_t1.mha
106_t2.mha | 106_t2.mha
107_t1.mha | 107_t1.mha
107_t2.mha | 107_t2.mha
108_t1.mha | 108_t1.mha
108_t2.mha | 108_t2.mha
109_t1.mha | 109_t1.mha
109_t2.mha | 109_t2.mha
10_t1.mha | 10_t1.mha
10_t2.mha | 10_t2.mha
110_t1.mha | 110_t1.mha
110_t2.mha | 110_t2.mha


## 5) Verificar pares axiales Al-Kafri/Sudirman

In [7]:
# ============================================
# 5) Pares axiales procesados
# ============================================
AXIAL_PAIRING_SRC = PFI_ROOT / "data/AXIAL_ALKAFRI/processed/pairing_v1"
print("AXIAL_PAIRING_SRC:", AXIAL_PAIRING_SRC, "| exists:", AXIAL_PAIRING_SRC.exists())

axial_pairs = []
for img in sorted(AXIAL_PAIRING_SRC.glob("*_image.npy")):
    mask = AXIAL_PAIRING_SRC / img.name.replace("_image.npy", "_mask.npy")
    if mask.exists():
        pair_id = img.name.replace("_image.npy", "")
        axial_pairs.append((pair_id, img, mask))

print("Pares axiales encontrados:", len(axial_pairs))
for pair_id, img, mask in axial_pairs[:20]:
    print(pair_id, "|", img.name, "|", mask.name)

if axial_pairs:
    arr = np.load(axial_pairs[0][1])
    msk = np.load(axial_pairs[0][2])
    print("\nPrimer axial image shape/dtype/range:", arr.shape, arr.dtype, float(np.nanmin(arr)), float(np.nanmax(arr)))
    print("Primer axial mask shape/dtype/range:", msk.shape, msk.dtype, float(np.nanmin(msk)), float(np.nanmax(msk)))


AXIAL_PAIRING_SRC: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/processed/pairing_v1 | exists: True
Pares axiales encontrados: 718
pair_0001 | pair_0001_image.npy | pair_0001_mask.npy
pair_0002 | pair_0002_image.npy | pair_0002_mask.npy
pair_0003 | pair_0003_image.npy | pair_0003_mask.npy
pair_0004 | pair_0004_image.npy | pair_0004_mask.npy
pair_0005 | pair_0005_image.npy | pair_0005_mask.npy
pair_0006 | pair_0006_image.npy | pair_0006_mask.npy
pair_0007 | pair_0007_image.npy | pair_0007_mask.npy
pair_0008 | pair_0008_image.npy | pair_0008_mask.npy
pair_0009 | pair_0009_image.npy | pair_0009_mask.npy
pair_0010 | pair_0010_image.npy | pair_0010_mask.npy
pair_0011 | pair_0011_image.npy | pair_0011_mask.npy
pair_0012 | pair_0012_image.npy | pair_0012_mask.npy
pair_0013 | pair_0013_image.npy | pair_0013_mask.npy
pair_0014 | pair_0014_image.npy | pair_0014_mask.npy
pair_0015 | pair_0015_image.npy | pair_0015_mask.npy
pair_0016 | pair_0016_image.npy | pair_0016_mask.npy
pair_0017 | pair

## 6) Selección del held-out

Por defecto usa 10 casos sagitales T2 y 30 pares axiales. Si luego encontrás un split limpio, reemplazá las listas `USER_SAG_CASE_NAMES` y `USER_AXIAL_PAIR_IDS`.


In [8]:
# ============================================
# 6) Selección del held-out
# ============================================
SELECTION_MODE = "preliminary_gt_not_confirmed_unseen"  # cambiar a clean_test_from_split si corresponde

USER_SAG_CASE_NAMES = None
USER_AXIAL_PAIR_IDS = None

DEFAULT_SAG_CASE_NAMES = [
    "101_t2.mha", "104_t2.mha", "105_t2.mha", "106_t2.mha", "107_t2.mha",
    "109_t2.mha", "113_t2.mha", "115_t2.mha", "116_t2.mha", "118_t2.mha",
]
DEFAULT_AXIAL_PAIR_IDS = [f"pair_{i:04d}" for i in range(1, 31)]

sag_case_names = USER_SAG_CASE_NAMES or DEFAULT_SAG_CASE_NAMES
axial_pair_ids = USER_AXIAL_PAIR_IDS or DEFAULT_AXIAL_PAIR_IDS

available_sag_by_name = {img.name: (img, mask) for img, mask in sag_pairs}
selected_sag = []
for name in sag_case_names:
    if name in available_sag_by_name:
        selected_sag.append((name, *available_sag_by_name[name]))

if len(selected_sag) < len(sag_case_names):
    already = {name for name, _, _ in selected_sag}
    for img, mask in sag_pairs:
        if img.name not in already:
            selected_sag.append((img.name, img, mask))
        if len(selected_sag) >= len(sag_case_names):
            break

available_ax_by_id = {pair_id: (img, mask) for pair_id, img, mask in axial_pairs}
selected_axial = []
for pair_id in axial_pair_ids:
    if pair_id in available_ax_by_id:
        selected_axial.append((pair_id, *available_ax_by_id[pair_id]))

if len(selected_axial) < len(axial_pair_ids):
    already = {pair_id for pair_id, _, _ in selected_axial}
    for pair_id, img, mask in axial_pairs:
        if pair_id not in already:
            selected_axial.append((pair_id, img, mask))
        if len(selected_axial) >= len(axial_pair_ids):
            break

print("SELECTION_MODE:", SELECTION_MODE)
print("Selected sagittal:", len(selected_sag))
for item in selected_sag:
    print(" -", item[0], "|", item[1], "|", item[2])

print("\nSelected axial:", len(selected_axial))
for item in selected_axial[:40]:
    print(" -", item[0], "|", item[1].name, "|", item[2].name)

if len(selected_sag) == 0:
    raise RuntimeError("No se seleccionó ningún caso sagital. Revisar rutas/máscaras SPIDER.")
if len(selected_axial) == 0:
    raise RuntimeError("No se seleccionó ningún caso axial. Revisar pairing_v1.")


SELECTION_MODE: preliminary_gt_not_confirmed_unseen
Selected sagittal: 10
 - 101_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/101_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/101_t2.mha
 - 104_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/104_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/104_t2.mha
 - 105_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/105_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/105_t2.mha
 - 106_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/106_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/106_t2.mha
 - 107_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/107_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/107_t2.mha
 - 109_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/109_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/109_t2.mha
 - 113_t2.mha 

## 7) Crear carpetas held-out locales

Copia desde Drive hacia `/content/pfi_qual_work/heldout`, no de Drive a Drive.


In [9]:
# ============================================
# 7) Crear carpetas held-out locales images/ + masks/
# ============================================
reset_dir(SAG_IMAGES_OUT)
reset_dir(SAG_MASKS_OUT)
reset_dir(AX_IMAGES_OUT)
reset_dir(AX_MASKS_OUT)

manifest_rows = []

for case_id, img, mask in selected_sag:
    img_dst = SAG_IMAGES_OUT / img.name
    mask_dst = SAG_MASKS_OUT / mask.name
    copy_pair(img, mask, img_dst, mask_dst)
    manifest_rows.append({
        "plane": "sagittal",
        "case_id": case_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest_local": str(img_dst),
        "mask_dest_local": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

for pair_id, img, mask in selected_axial:
    img_dst = AX_IMAGES_OUT / f"{pair_id}.npy"
    mask_dst = AX_MASKS_OUT / f"{pair_id}.npy"
    copy_pair(img, mask, img_dst, mask_dst)
    manifest_rows.append({
        "plane": "axial",
        "case_id": pair_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest_local": str(img_dst),
        "mask_dest_local": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

print_tree_counts()

manifest_df = pd.DataFrame(manifest_rows)
manifest_csv = LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.csv"
manifest_json = LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.json"

manifest_df.to_csv(manifest_csv, index=False)
manifest_json.write_text(json.dumps({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_mode": SELECTION_MODE,
    "warning": "Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con ground truth, no como test limpio final.",
    "sagittal_test_dir_local": str(SAG_TEST_DIR),
    "axial_test_dir_local": str(AX_TEST_DIR),
    "counts": {"sagittal": len(selected_sag), "axial": len(selected_axial)},
    "rows": manifest_rows,
}, indent=2, ensure_ascii=False), encoding="utf-8")

print("Manifest CSV local:", manifest_csv)
print("Manifest JSON local:", manifest_json)
manifest_df.head()


Copy failed attempt 1/3: /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/107_t2.mha -> /content/pfi_qual_work/heldout/sagittal/images/107_t2.mha | OSError(107, 'Transport endpoint is not connected')
Intentando force_remount de Google Drive...
Mounted at /content/drive
Sagital images: 10
Sagital masks: 10
Axial images: 30
Axial masks: 30
Manifest CSV local: /content/pfi_qual_work/reports/qual-002-heldout-manifest.csv
Manifest JSON local: /content/pfi_qual_work/reports/qual-002-heldout-manifest.json


,plane,case_id,image_source,mask_source,image_dest_local,mask_dest_local,selection_mode
0,sagittal,101_t2.mha,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/pfi_qual_work/heldout/sagittal/images...,/content/pfi_qual_work/heldout/sagittal/masks/...,preliminary_gt_not_confirmed_unseen
1,sagittal,104_t2.mha,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/pfi_qual_work/heldout/sagittal/images...,/content/pfi_qual_work/heldout/sagittal/masks/...,preliminary_gt_not_confirmed_unseen
2,sagittal,105_t2.mha,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/pfi_qual_work/heldout/sagittal/images...,/content/pfi_qual_work/heldout/sagittal/masks/...,preliminary_gt_not_confirmed_unseen
3,sagittal,106_t2.mha,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/pfi_qual_work/heldout/sagittal/images...,/content/pfi_qual_work/heldout/sagittal/masks/...,preliminary_gt_not_confirmed_unseen
4,sagittal,107_t2.mha,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...,/content/pfi_qual_work/heldout/sagittal/images...,/content/pfi_qual_work/heldout/sagittal/masks/...,preliminary_gt_not_confirmed_unseen


## 8) Resolver arquitectura de modelos

Esta versión no requiere `scripts/evaluate_model.py`.

Primero intenta importar `build_checkpoint_model()` desde `pfi_ai_service.model_architectures`. Si no lo encuentra, se detiene con una instrucción clara.


In [12]:
%cd /content
!rm -rf PFI_MVPTest_Enzo_AImodule
!git clone https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git
!find /content/PFI_MVPTest_Enzo_AImodule -name "model_architectures.py" -print
!find /content/PFI_MVPTest_Enzo_AImodule -name "evaluate_model.py" -print


/content
Cloning into 'PFI_MVPTest_Enzo_AImodule'...
remote: Enumerating objects: 1466, done.
remote: Counting objects: 100% (299/299), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 1466 (delta 198), reused 141 (delta 76), pack-reused 1167 (from 2)
Receiving objects: 100% (1466/1466), 13.64 MiB | 18.97 MiB/s, done.
Resolving deltas: 100% (885/885), done.
/content/PFI_MVPTest_Enzo_AImodule/ai_service/pfi_ai_service/model_architectures.py
/content/PFI_MVPTest_Enzo_AImodule/scripts/evaluate_model.py


In [13]:
from pathlib import Path

print("Buscando model_architectures.py...")
!find /content/drive/MyDrive/PFI_MVP -name "model_architectures.py" -print
!find /content -name "model_architectures.py" -print

Buscando model_architectures.py...
/content/PFI_MVPTest_Enzo_AImodule/ai_service/pfi_ai_service/model_architectures.py


In [14]:
# ============================================
# 8) Resolver arquitectura de modelos
# ============================================
import torch

AI_SERVICE_CANDIDATES = [
    PFI_ROOT / "repo/ai_service",
    PFI_ROOT / "PFI_MVPTest_Enzo_AImodule/ai_service",
    DRIVE_ROOT / "PFI_MVPTest_Enzo_AImodule/ai_service",
    Path("/content/PFI_MVPTest_Enzo_AImodule/ai_service"),
    Path.cwd() / "ai_service",
]

ai_service_path = None
for candidate in AI_SERVICE_CANDIDATES:
    if (candidate / "pfi_ai_service/model_architectures.py").exists():
        ai_service_path = candidate
        break

print("ai_service_path:", ai_service_path)

if ai_service_path is None:
    print("No encontré pfi_ai_service/model_architectures.py en los candidatos:")
    for c in AI_SERVICE_CANDIDATES:
        print(" -", c, "| exists:", c.exists())
    raise RuntimeError(
        "No encontré model_architectures.py. Montá o cloná el AI Module en Drive, "
        "o copiá pfi_ai_service/model_architectures.py a /content/drive/MyDrive/PFI_MVP/repo/ai_service."
    )

sys.path.insert(0, str(ai_service_path))
from pfi_ai_service.model_architectures import build_checkpoint_model

print("Import OK: build_checkpoint_model desde", ai_service_path)
print("torch:", torch.__version__)


ai_service_path: /content/PFI_MVPTest_Enzo_AImodule/ai_service
Import OK: build_checkpoint_model desde /content/PFI_MVPTest_Enzo_AImodule/ai_service
torch: 2.11.0+cpu


## 9) Evaluador self-contained

Calcula Dice/IoU por clase, macro foreground y variantes útiles. Usa el mismo `build_checkpoint_model()` del AI Module, pero no depende de `scripts/evaluate_model.py`.


In [15]:
# ============================================
# 9) Evaluador interno QUAL-004
# ============================================
from PIL import Image
import SimpleITK as sitk

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

def torch_load(path: Path):
    return torch.load(str(path), map_location="cpu", weights_only=False)


def checkpoint_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint
    raise ValueError("No se pudo detectar state_dict en el checkpoint.")


def robust_normalize(arr, p_low=1, p_high=99):
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        mn, mx = float(np.min(arr)), float(np.max(arr))
        if mx <= mn:
            return np.zeros_like(arr, dtype=np.float32)
        lo, hi = mn, mx
    out = np.clip(arr, lo, hi)
    return ((out - lo) / (hi - lo)).astype(np.float32)


def resize_image_2d(arr, target_size=256):
    img = Image.fromarray(np.asarray(arr, dtype=np.float32))
    img = img.resize((target_size, target_size), resample=Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)


def resize_mask_2d(mask, target_size=256):
    img = Image.fromarray(np.asarray(mask).astype(np.int32), mode="I")
    img = img.resize((target_size, target_size), resample=Image.NEAREST)
    return np.asarray(img, dtype=np.int64)


def read_mha(path: Path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return np.asarray(arr)


def read_any_image(path: Path):
    path = Path(path)
    if path.suffix.lower() == ".npy":
        return np.load(path, allow_pickle=False)
    if path.suffix.lower() in [".mha", ".mhd"]:
        return read_mha(path)
    if path.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
        return np.asarray(Image.open(path).convert("F"), dtype=np.float32)
    raise ValueError(f"Formato no soportado: {path}")


def extract_sagittal_pair(image_volume, mask_volume, axis=2):
    img = np.squeeze(np.asarray(image_volume))
    msk = np.squeeze(np.asarray(mask_volume))
    if img.ndim == 2:
        return img, msk
    if img.ndim != 3 or msk.ndim != 3:
        raise ValueError(f"Sagital esperaba 2D/3D. image={img.shape}, mask={msk.shape}")

    axis = int(axis)
    if axis < 0 or axis >= img.ndim:
        axis = int(np.argmin(img.shape))

    idx = img.shape[axis] // 2
    return np.take(img, idx, axis=axis), np.take(msk, idx, axis=axis)


def extract_axial_pair(image_arr, mask_arr):
    img = np.squeeze(np.asarray(image_arr))
    msk = np.squeeze(np.asarray(mask_arr))
    if img.ndim == 2:
        return img, msk
    if img.ndim == 3:
        idx = img.shape[0] // 2
        return img[idx], msk[idx]
    raise ValueError(f"Axial esperaba 2D/3D. image={img.shape}, mask={msk.shape}")


def map_sagittal_mask(mask_raw, checkpoint):
    mask = np.asarray(mask_raw, dtype=np.int64)

    # Si ya está en 0..3, no remapear.
    uniques = set(np.unique(mask).astype(int).tolist())
    if uniques.issubset({0, 1, 2, 3}):
        return mask.astype(np.int64)

    out = np.zeros_like(mask, dtype=np.int64)

    mapping = checkpoint.get("label_group_mapping") if isinstance(checkpoint, dict) else None
    if isinstance(mapping, dict):
        # Intento flexible: soporta claves por nombre o por índice.
        def values_to_set(v):
            if isinstance(v, dict):
                vals = []
                for vv in v.values():
                    if isinstance(vv, (list, tuple, set)):
                        vals.extend(vv)
                    else:
                        vals.append(vv)
                return set(int(x) for x in vals if str(x).lstrip("-").isdigit())
            if isinstance(v, (list, tuple, set)):
                return set(int(x) for x in v if str(x).lstrip("-").isdigit())
            if str(v).lstrip("-").isdigit():
                return {int(v)}
            return set()

        for k, v in mapping.items():
            lk = str(k).lower()
            vals = values_to_set(v)
            if not vals:
                continue
            if "vertebra" in lk or lk in ["1", "vertebra_group"]:
                out[np.isin(mask, list(vals))] = 1
            elif "canal" in lk or lk in ["2", "spinal_canal"]:
                out[np.isin(mask, list(vals))] = 2
            elif "disc" in lk or "disco" in lk or lk in ["3", "disc_group"]:
                out[np.isin(mask, list(vals))] = 3

        if np.max(out) > 0:
            return out

    # Fallback SPIDER conocido:
    # 1..99 vértebras, 100 canal, 200+ discos.
    out[(mask >= 1) & (mask < 100)] = 1
    out[mask == 100] = 2
    out[mask >= 200] = 3
    return out.astype(np.int64)


def map_axial_mask(mask_raw):
    mask = np.asarray(mask_raw, dtype=np.int64)
    uniques = set(np.unique(mask).astype(int).tolist())
    if uniques.issubset({0, 1, 2, 3, 4, 5}):
        return mask.astype(np.int64)

    out = np.zeros_like(mask, dtype=np.int64)
    # Al-Kafri/Sudirman pairing_v1: background=250, clases crudas 0/50/100/150/200.
    out[mask == 250] = 0
    out[mask == 0] = 1
    out[mask == 50] = 2
    out[mask == 100] = 3
    out[mask == 150] = 4
    out[mask == 200] = 5
    return out.astype(np.int64)


def prepare_pair_for_model(image_raw, mask_raw, plane, checkpoint, target_size=256):
    if plane == "sagittal":
        axis = int(checkpoint.get("sagittal_axis", 2)) if isinstance(checkpoint, dict) else 2
        image_2d, mask_2d = extract_sagittal_pair(image_raw, mask_raw, axis=axis)
        mask_2d = map_sagittal_mask(mask_2d, checkpoint)
    elif plane == "axial":
        image_2d, mask_2d = extract_axial_pair(image_raw, mask_raw)
        mask_2d = map_axial_mask(mask_2d)
    else:
        raise ValueError(plane)

    image_2d = robust_normalize(image_2d)
    image_2d = resize_image_2d(image_2d, target_size=target_size)
    mask_2d = resize_mask_2d(mask_2d, target_size=target_size)

    return image_2d.astype(np.float32), mask_2d.astype(np.int64)


def dice_iou_for_class(pred, gt, class_id):
    pred_c = (pred == class_id)
    gt_c = (gt == class_id)
    pred_sum = int(pred_c.sum())
    gt_sum = int(gt_c.sum())
    inter = int(np.logical_and(pred_c, gt_c).sum())
    union = int(np.logical_or(pred_c, gt_c).sum())

    if pred_sum + gt_sum == 0:
        dice = 1.0
    else:
        dice = 2.0 * inter / (pred_sum + gt_sum)

    if union == 0:
        iou = 1.0
    else:
        iou = inter / union

    return {
        "dice": float(dice),
        "iou": float(iou),
        "pred_pixels": pred_sum,
        "gt_pixels": gt_sum,
        "intersection_pixels": inter,
        "union_pixels": union,
        "gt_present": gt_sum > 0,
        "pred_present": pred_sum > 0,
    }


def list_test_pairs(test_dir: Path):
    images_dir = test_dir / "images"
    masks_dir = test_dir / "masks"

    image_files = sorted([p for p in images_dir.iterdir() if p.is_file()])
    pairs = []
    for img in image_files:
        mask = masks_dir / img.name
        if mask.exists():
            pairs.append((img.stem, img, mask))
        else:
            # fallback por stem con cualquier extensión
            cands = sorted(masks_dir.glob(img.stem + ".*"))
            if cands:
                pairs.append((img.stem, img, cands[0]))
            else:
                print("Sin máscara para:", img)
    return pairs


def evaluate_internal(plane, checkpoint_path, test_dir, num_classes, output_json):
    checkpoint = torch_load(checkpoint_path)
    model_id = "sagittal_spider" if plane == "sagittal" else "axial_t2_alkafri"
    model = build_checkpoint_model(model_id, checkpoint)
    model.to(DEVICE)
    model.eval()

    pairs = list_test_pairs(Path(test_dir))
    if not pairs:
        raise RuntimeError(f"No hay pares image/mask en {test_dir}")

    class_names = (
        ["background", "vertebra_group", "canal", "disc_group"]
        if plane == "sagittal"
        else ["background_250", "raw_0", "raw_50", "raw_100", "raw_150", "raw_200"]
    )

    per_case = []
    accum = {c: [] for c in range(num_classes)}

    with torch.inference_mode():
        for case_id, img_path, mask_path in pairs:
            image_raw = read_any_image(img_path)
            mask_raw = read_any_image(mask_path)

            image_2d, gt = prepare_pair_for_model(
                image_raw=image_raw,
                mask_raw=mask_raw,
                plane=plane,
                checkpoint=checkpoint,
                target_size=TARGET_SIZE,
            )

            x = torch.from_numpy(image_2d[None, None, :, :].astype(np.float32)).to(DEVICE)
            logits = model(x)
            pred = torch.argmax(logits, dim=1).detach().cpu().numpy()[0].astype(np.int64)

            case_metrics = {
                "case_id": case_id,
                "image": str(img_path),
                "mask": str(mask_path),
                "image_prepared_shape": list(image_2d.shape),
                "mask_prepared_shape": list(gt.shape),
                "classes": {},
            }

            for c in range(num_classes):
                m = dice_iou_for_class(pred, gt, c)
                accum[c].append(m)
                cname = class_names[c] if c < len(class_names) else f"class_{c}"
                case_metrics["classes"][str(c)] = {"name": cname, **m}

            per_case.append(case_metrics)

    by_class = {}
    for c in range(num_classes):
        rows = accum[c]
        cname = class_names[c] if c < len(class_names) else f"class_{c}"
        by_class[str(c)] = {
            "name": cname,
            "dice_mean": float(np.mean([r["dice"] for r in rows])),
            "iou_mean": float(np.mean([r["iou"] for r in rows])),
            "gt_pixels_total": int(sum(r["gt_pixels"] for r in rows)),
            "pred_pixels_total": int(sum(r["pred_pixels"] for r in rows)),
            "gt_present_cases": int(sum(1 for r in rows if r["gt_present"])),
            "pred_present_cases": int(sum(1 for r in rows if r["pred_present"])),
            "n_cases": len(rows),
        }

    fg_ids = list(range(1, num_classes))
    dice_macro_foreground = float(np.mean([by_class[str(c)]["dice_mean"] for c in fg_ids]))
    iou_macro_foreground = float(np.mean([by_class[str(c)]["iou_mean"] for c in fg_ids]))

    present_fg_ids = [c for c in fg_ids if by_class[str(c)]["gt_present_cases"] > 0]
    dice_macro_foreground_present_gt = float(np.mean([by_class[str(c)]["dice_mean"] for c in present_fg_ids])) if present_fg_ids else None
    iou_macro_foreground_present_gt = float(np.mean([by_class[str(c)]["iou_mean"] for c in present_fg_ids])) if present_fg_ids else None

    result = {
        "qual_id": "QUAL-004",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "evaluator": "notebook_36_v3_self_contained",
        "plane": plane,
        "checkpoint": str(checkpoint_path),
        "checkpoint_sha256": sha256_file(Path(checkpoint_path)),
        "test_dir": str(test_dir),
        "selection_mode": SELECTION_MODE,
        "warning": "Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con ground truth, no como test limpio final.",
        "target_size": TARGET_SIZE,
        "num_classes": num_classes,
        "n_cases": len(pairs),
        "class_names": class_names,
        "metrics": {
            "dice_macro_foreground": dice_macro_foreground,
            "iou_macro_foreground": iou_macro_foreground,
            "dice_macro_foreground_present_gt": dice_macro_foreground_present_gt,
            "iou_macro_foreground_present_gt": iou_macro_foreground_present_gt,
            "dice_by_class": {k: v["dice_mean"] for k, v in by_class.items()},
            "iou_by_class": {k: v["iou_mean"] for k, v in by_class.items()},
            "by_class": by_class,
        },
        "per_case": per_case,
    }

    if plane == "axial" and num_classes >= 6:
        useful_ids = [2, 3, 4, 5]
        result["metrics"]["dice_macro_excluding_raw0"] = float(np.mean([by_class[str(c)]["dice_mean"] for c in useful_ids]))
        result["metrics"]["iou_macro_excluding_raw0"] = float(np.mean([by_class[str(c)]["iou_mean"] for c in useful_ids]))
        result["metrics"]["note_raw0"] = "raw_0 se reporta por transparencia, pero también se informa macro excluyendo raw_0 por ser clase minoritaria/intermitente."

    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    return result


DEVICE: cpu


## 10) Ejecutar QUAL-004

Esta celda reemplaza la llamada a `scripts/evaluate_model.py`.


In [17]:
# ============================================
# PATCH: adaptar build_checkpoint_model si devuelve tupla
# ============================================

_original_build_checkpoint_model = build_checkpoint_model

def unwrap_model(candidate):
    """
    Acepta:
    - nn.Module directo
    - tuple/list donde uno de los elementos es nn.Module
    - dict donde algún valor es nn.Module
    """
    if hasattr(candidate, "to") and hasattr(candidate, "eval"):
        return candidate

    if isinstance(candidate, (tuple, list)):
        for item in candidate:
            if hasattr(item, "to") and hasattr(item, "eval"):
                return item

    if isinstance(candidate, dict):
        for item in candidate.values():
            if hasattr(item, "to") and hasattr(item, "eval"):
                return item

    raise TypeError(
        f"No pude extraer un modelo PyTorch desde build_checkpoint_model. "
        f"Tipo recibido: {type(candidate)}"
    )

def build_checkpoint_model(model_id, checkpoint):
    result = _original_build_checkpoint_model(model_id, checkpoint)
    model = unwrap_model(result)
    return model

print("Patch OK: build_checkpoint_model ahora devuelve solo el modelo nn.Module.")

Patch OK: build_checkpoint_model ahora devuelve solo el modelo nn.Module.


In [18]:
# ============================================
# 10) Ejecutar evaluación self-contained
# ============================================
sag_out_json = LOCAL_DOCS_ROOT / "qual-004-sagittal.json"
ax_out_json  = LOCAL_DOCS_ROOT / "qual-004-axial.json"

sag_log = LOCAL_DOCS_ROOT / "qual-004-sagittal.log"
ax_log  = LOCAL_DOCS_ROOT / "qual-004-axial.log"

try:
    sag_result = evaluate_internal(
        plane="sagittal",
        checkpoint_path=SAGITTAL_CKPT,
        test_dir=SAG_TEST_DIR,
        num_classes=4,
        output_json=sag_out_json,
    )
    sag_log.write_text("OK - evaluación sagital ejecutada por notebook_36_v3_self_contained\n", encoding="utf-8")
    print("Sagital OK:", sag_out_json)
except Exception as e:
    sag_log.write_text(repr(e), encoding="utf-8")
    raise

try:
    ax_result = evaluate_internal(
        plane="axial",
        checkpoint_path=AXIAL_CKPT,
        test_dir=AX_TEST_DIR,
        num_classes=6,
        output_json=ax_out_json,
    )
    ax_log.write_text("OK - evaluación axial ejecutada por notebook_36_v3_self_contained\n", encoding="utf-8")
    print("Axial OK:", ax_out_json)
except Exception as e:
    ax_log.write_text(repr(e), encoding="utf-8")
    raise

print("\nEvaluación finalizada.")
print("Sagital:", sag_out_json)
print("Axial:", ax_out_json)


/tmp/ipykernel_1170/846809945.py:44: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(np.asarray(mask).astype(np.int32), mode="I")


Sagital OK: /content/pfi_qual_work/docs/qual-004-sagittal.json
Axial OK: /content/pfi_qual_work/docs/qual-004-axial.json

Evaluación finalizada.
Sagital: /content/pfi_qual_work/docs/qual-004-sagittal.json
Axial: /content/pfi_qual_work/docs/qual-004-axial.json


## 11) Mostrar JSONs y generar resumen para Claude/Codex

In [19]:
# ============================================
# 11) Mostrar resultados y generar resumen
# ============================================
def load_json(path: Path):
    if not path.exists():
        print("NO EXISTE:", path)
        return None
    return json.loads(path.read_text(encoding="utf-8"))


sag_result = load_json(sag_out_json)
ax_result = load_json(ax_out_json)

print("\n==== qual-004-sagittal.json ====")
print(json.dumps(sag_result, indent=2, ensure_ascii=False)[:12000] if sag_result is not None else "NO JSON")

print("\n==== qual-004-axial.json ====")
print(json.dumps(ax_result, indent=2, ensure_ascii=False)[:12000] if ax_result is not None else "NO JSON")

sag_macro = sag_result["metrics"]["dice_macro_foreground"] if sag_result else None
ax_macro = ax_result["metrics"]["dice_macro_foreground"] if ax_result else None
ax_macro_excl_raw0 = ax_result["metrics"].get("dice_macro_excluding_raw0") if ax_result else None

summary_md = f'''# QUAL-004 - Evaluación preliminar con Ground Truth

Fecha UTC: {datetime.now(timezone.utc).isoformat()}

## Advertencia metodológica

selection_mode: `{SELECTION_MODE}`

Si `selection_mode` no es `clean_test_from_split`, estos números deben reportarse como **evaluación preliminar con ground truth**, no como test limpio final por paciente.

## Inputs

Sagital:
- test_dir local: `{SAG_TEST_DIR}`
- checkpoint: `{SAGITTAL_CKPT}`
- casos: {len(selected_sag)}

Axial:
- test_dir local: `{AX_TEST_DIR}`
- checkpoint: `{AXIAL_CKPT}`
- casos: {len(selected_axial)}

## Outputs locales

- `{sag_out_json}`
- `{ax_out_json}`
- `{sag_log}`
- `{ax_log}`
- `{manifest_json}`

## Métricas principales detectadas

- Sagital dice macro foreground/no-bg: `{sag_macro}`
- Axial dice macro foreground/no-bg: `{ax_macro}`
- Axial dice macro excluyendo raw_0: `{ax_macro_excl_raw0}`

## Nota para Claude/Codex

Los JSON completos están guardados en `qual-004-sagittal.json` y `qual-004-axial.json`.

Esta corrida fue ejecutada con el evaluador interno del notebook 36 v3 porque en Colab no se encontró `scripts/evaluate_model.py`.

Usar estos archivos como evidencia preliminar para QUAL-005, sin presentarlos como test limpio final si no se confirma split limpio por paciente.
'''.strip()

summary_path = LOCAL_DOCS_ROOT / "qual-004-summary.md"
summary_path.write_text(summary_md, encoding="utf-8")

print("\n==== SUMMARY PARA CLAUDE/CODEX ====")
print(summary_md)
print("\nGuardado local:", summary_path)



==== qual-004-sagittal.json ====
{
  "qual_id": "QUAL-004",
  "created_at_utc": "2026-07-16T20:55:44.699482+00:00",
  "evaluator": "notebook_36_v3_self_contained",
  "plane": "sagittal",
  "checkpoint": "/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt",
  "checkpoint_sha256": "7dd393cc750311c98003516d8110136310c31e8b6f0f00b6815f949fd61ef15b",
  "test_dir": "/content/pfi_qual_work/heldout/sagittal",
  "selection_mode": "preliminary_gt_not_confirmed_unseen",
  "warning": "Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con ground truth, no como test limpio final.",
  "target_size": 256,
  "num_classes": 4,
  "n_cases": 10,
  "class_names": [
    "background",
    "vertebra_group",
    "canal",
    "disc_group"
  ],
  "metrics": {
    "dice_macro_foreground": 0.10134680926498953,
    "iou_macro_foreground": 0.1006818092068736,
    "dice_macro_foreground_present_gt": 0.0040404277949686145,
    "iou_macro_foreground_prese

## 12) Sincronizar evidencia a Drive y comprimir

In [20]:
# ============================================
# 12) Sincronizar evidencia a Drive y comprimir
# ============================================
files_to_sync = [
    LOCAL_REPORTS_ROOT / "qual-002-split-candidates.txt",
    LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.csv",
    LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.json",
    LOCAL_DOCS_ROOT / "qual-004-sagittal.json",
    LOCAL_DOCS_ROOT / "qual-004-axial.json",
    LOCAL_DOCS_ROOT / "qual-004-sagittal.log",
    LOCAL_DOCS_ROOT / "qual-004-axial.log",
    LOCAL_DOCS_ROOT / "qual-004-summary.md",
]

synced = []
for p in files_to_sync:
    if not p.exists():
        print("No existe local, no se sincroniza:", p)
        continue
    dst = (DRIVE_REPORTS_ROOT if p.parent == LOCAL_REPORTS_ROOT else DRIVE_DOCS_ROOT) / p.name
    copy_file_robust(p, dst)
    synced.append(dst)
    print("Sincronizado:", p, "->", dst)

local_zip_path = LOCAL_REPORTS_ROOT / "qual-002-004-evidence.zip"
zip_path = DRIVE_REPORTS_ROOT / "qual-002-004-evidence.zip"

with zipfile.ZipFile(local_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in files_to_sync:
        if p.exists():
            z.write(p, arcname=p.name)

copy_file_robust(local_zip_path, zip_path)

print("ZIP local:", local_zip_path)
print("ZIP Drive:", zip_path)
print("ZIP exists:", zip_path.exists())
if zip_path.exists():
    print("ZIP size MB:", round(zip_path.stat().st_size / (1024 * 1024), 3))
    print("ZIP sha256:", sha256_file(zip_path))


Sincronizado: /content/pfi_qual_work/reports/qual-002-split-candidates.txt -> /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-split-candidates.txt
Sincronizado: /content/pfi_qual_work/reports/qual-002-heldout-manifest.csv -> /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-heldout-manifest.csv
Sincronizado: /content/pfi_qual_work/reports/qual-002-heldout-manifest.json -> /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-heldout-manifest.json
Sincronizado: /content/pfi_qual_work/docs/qual-004-sagittal.json -> /content/drive/MyDrive/PFI_MVP/docs/qual-004-sagittal.json
Sincronizado: /content/pfi_qual_work/docs/qual-004-axial.json -> /content/drive/MyDrive/PFI_MVP/docs/qual-004-axial.json
Sincronizado: /content/pfi_qual_work/docs/qual-004-sagittal.log -> /content/drive/MyDrive/PFI_MVP/docs/qual-004-sagittal.log
Sincronizado: /content/pfi_qual_work/docs/qual-004-axial.log -> /content/drive/MyDrive/PFI_MVP/docs/qual-004-axial.log
Sincronizado: /content/pfi_qual_work/docs/qual-004

## 13) Mensaje sugerido para Claude/Codex

In [21]:
print(f'''
QUAL-002 / QUAL-004 ejecutado en Colab.

Held-out:
- selection_mode: {SELECTION_MODE}
- sagital casos: {len(selected_sag)}
- axial casos: {len(selected_axial)}

Carpetas usadas por el evaluador:
- sagital test_dir local: {SAG_TEST_DIR}
- axial test_dir local: {AX_TEST_DIR}

Outputs en Drive:
- {DRIVE_DOCS_ROOT / "qual-004-sagittal.json"}
- {DRIVE_DOCS_ROOT / "qual-004-axial.json"}
- {DRIVE_DOCS_ROOT / "qual-004-summary.md"}
- {DRIVE_REPORTS_ROOT / "qual-002-heldout-manifest.json"}
- {DRIVE_REPORTS_ROOT / "qual-002-004-evidence.zip"}

Aclaración:
Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con GT,
no como test limpio final por paciente.

Nota:
La corrida usó el evaluador interno del notebook 36 v3 porque Colab no encontró scripts/evaluate_model.py.

Copio a continuación el summary y/o los dos JSON.
''')



QUAL-002 / QUAL-004 ejecutado en Colab.

Held-out:
- selection_mode: preliminary_gt_not_confirmed_unseen
- sagital casos: 10
- axial casos: 30

Carpetas usadas por el evaluador:
- sagital test_dir local: /content/pfi_qual_work/heldout/sagittal
- axial test_dir local: /content/pfi_qual_work/heldout/axial

Outputs en Drive:
- /content/drive/MyDrive/PFI_MVP/docs/qual-004-sagittal.json
- /content/drive/MyDrive/PFI_MVP/docs/qual-004-axial.json
- /content/drive/MyDrive/PFI_MVP/docs/qual-004-summary.md
- /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-heldout-manifest.json
- /content/drive/MyDrive/PFI_MVP/qual/reports/qual-002-004-evidence.zip

Aclaración:
Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con GT,
no como test limpio final por paciente.

Nota:
La corrida usó el evaluador interno del notebook 36 v3 porque Colab no encontró scripts/evaluate_model.py.

Copio a continuación el summary y/o los dos JSON.

